# 450 — Predefined time-frequency ROI pooling

The **cluster-blind, condition-blind** pooling path. A fixed, physiology-driven library of
time-frequency **ROIs** (`functions/roi_config.py`) is applied to **every** electrode × condition
ERSP — independent of the 02 clusters and of audio/picture/reading. Each ROI is a 2-D box
(`f_rows × t_bins`, 1-based inclusive) + a **sign** hypothesis, encoding a documented marker
(sensory-onset HGA, sustained processing, pre-response planning, motor execution, alpha/beta ERD,
theta retrieval, …). Pooling turns each ERSP into a small citation-anchored feature vector.

- **Pool** = mean dB in the box. **Qualify** = the box clears the clustering σ/proportion gate in
  the ROI's **own sign** (`pos`/`neg`/`both`).
- **Option A aggregation:** a contact **expresses** an ROI if it qualifies in **≥1 condition**.
- This is **additive** — the zone-based `410–490` path is untouched.

> ⚠️ Two predeterminant issues to reconcile (see the validation table in §1): the **ds vs full**
> Hz ranges disagree for `alpha_beta_suppression` and `broadband_activation_index`; and the
> `broadband_activation_index` box-mean is **not** Manning's broadband index (a slope/offset
> measure) — treat it cautiously or redefine it.


In [ ]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display, Markdown, Image
sys.path.insert(0, str(Path('..').resolve()))
from functions import lf_pool as P

INPUT_DIR = Path(r'\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\outputs\04_ersp_LM_RAWONLY')
if not INPUT_DIR.exists():
    INPUT_DIR = Path('../01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY').resolve()

print('INPUT_DIR :', INPUT_DIR, '| exists:', INPUT_DIR.exists())
print('OUTPUTS   :', P.OUTPUTS_ROOT)
print('COORDS    :', P.COORDS_DIR, '| exists:', P.COORDS_DIR.exists())
print('conditions:', P.CONDITIONS, '| zones:', P.DEFAULT_ZONES)
print('feature sets:', P.FEATURE_SETS, '| window shapes:', P.WINDOW_SHAPES)


In [ ]:
# ---- knobs ----
USE_DS       = True                  # True: 15x30 ds grid · False: full 129x300
GRID         = 'ds' if USE_DS else 'full'
DS_TIME_BINS = 30
print('grid:', GRID, '| n ROIs:', len(P.ROI_PARAMS[GRID]))


## 1 — The ROI map (with legend) + consistency check
Each box labelled `(a) (b) (c) …`, coloured red = positive / blue = negative / purple = both.
Same-box opposite-sign ROIs overlap, so positive and negative signs are also drawn separately.
The validation table flags any ds↔full Hz / sign mismatches.


In [ ]:
import matplotlib.pyplot as plt
disc = P.new_run_dir('roi', GRID + '_map')
for g in ('ds', 'full'):
    P.plot_roi_map(grid=g, out_png=disc / f'roi_map_{g}.png'); plt.show()
# pos / neg separated (so the early-onset vs early-suppression twins don't overlap)
P.plot_roi_map(grid=GRID, signs=('pos', 'both')); plt.title('positive / both'); plt.show()
P.plot_roi_map(grid=GRID, signs=('neg',));         plt.title('negative');       plt.show()
display(P.roi_legend(grid=GRID))
print('\nds <-> full consistency (hz_mismatch / sign_mismatch = reconcile):')
display(P.validate_roi_config())


## 2 — Pool every contact against every ROI
Condition-blind: one row per (contact, condition, ROI). Cached to
`outputs/_dataset/pooling/roi_table_<grid>.parquet`.


In [ ]:
df_meta, X_full = P.prepare_pooling_dataset(INPUT_DIR)
X = P.downsample_dataset(X_full, time_bins=DS_TIME_BINS) if USE_DS else X_full
df_roi = P.build_roi_table(df_meta, X, grid=GRID)
print('roi table:', df_roi.shape)
df_roi.head()


## 3 — Per-ROI qualifier counts (option A)
Distinct contacts expressing each ROI in ≥1 condition.


In [ ]:
counts = P.roi_counts(df_roi)
display(counts)
ax = counts.set_index('roi_tag')['n_contacts'].plot.barh(
    figsize=(8, 0.4 * len(counts) + 1))
ax.set_xlabel('# contacts expressing'); ax.set_title('ROI expression (option A: ≥1 condition)')
ax.invert_yaxis(); plt.tight_layout(); plt.show()


## 4 — Where do they map? (ROI × Yeo-7)
Crosstab of expressing contacts by anatomical network — the condition-blind "where each signature
lives" map. (Yeo from the coords CSVs; no MNE needed.)


In [ ]:
coords = P.load_coords()
ct = P.roi_region_crosstab(df_roi, coords, scheme='yeo7')
display(ct)
